In [1]:
import os
os.makedirs("ML_FINAL/data", exist_ok=True)
os.makedirs("ML_FINAL/results", exist_ok=True)

In [2]:
!pip install sentence-transformers pandas scikit-learn

In [3]:
import pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import os

jobs_path = "ML_FINAL/data/jobs.csv"
resumes_path = "ML_FINAL/data/resume_variants.csv"

jobs = pd.read_csv(jobs_path)
resumes = pd.read_csv(resumes_path)

print("Jobs shape:", jobs.shape)
print("Resume variants shape:", resumes.shape)

display(jobs.head())
display(resumes.head())

Jobs shape: (5, 5)
Resume variants shape: (40, 8)


,job_id,domain,title,company_name,job_description
0,J1,Software Engineering,Software Engineer,GOYT,Job Description:GOYT is seeking a skilled and ...
1,J2,Finance,Financial Analyst,Aya Healthcare,Aya Healthcare has an exciting 26-week contrac...
2,J3,Marketing,Marketing Coordinator,Corcoran Sawyer Smith,Job descriptionA leading real estate firm in N...
3,J4,Healthcare,Clinical Data Analyst,Insight Global,Must Have: 2+ years of current experience in a...
4,J5,Education,Academic Advisor,Kennesaw State University,About Us Are you ready to join a community lea...


,resume_id,domain,version,changed_signal,name,pronouns,university,resume_text
0,R1,Software Engineering,original,none,Emily Johnson,she/her,Stanford University,"Emily Johnson, she/her. B.S. in Computer Scien..."
1,R1,Software Engineering,name_changed,name,Aisha Khan,she/her,Stanford University,"Aisha Khan, she/her. B.S. in Computer Science ..."
2,R1,Software Engineering,pronoun_changed,pronoun,Emily Johnson,he/him,Stanford University,"Emily Johnson, he/him. B.S. in Computer Scienc..."
3,R1,Software Engineering,university_changed,university,Emily Johnson,she/her,Morgan State University,"Emily Johnson, she/her. B.S. in Computer Scien..."
4,R2,Software Engineering,original,none,Michael Smith,he/him,Massachusetts Institute of Technology,"Michael Smith, he/him. B.S. in Software Engine..."


In [4]:
# Loading SBERT model
model = SentenceTransformer("all-MiniLM-L6-v2")
results = []
# Comparing every resume variant with every job description
for _, resume_row in resumes.iterrows():
    resume_text = str(resume_row["resume_text"])
    for _, job_row in jobs.iterrows():
        job_text = str(job_row["job_description"])
        # Creating embeddings
        resume_embedding = model.encode([resume_text])
        job_embedding = model.encode([job_text])
        # Cosine similarity score
        score = cosine_similarity(resume_embedding, job_embedding)[0][0]
        results.append({
            "resume_id": resume_row["resume_id"],
            "resume_domain": resume_row["domain"],
            "version": resume_row["version"],
            "changed_signal": resume_row["changed_signal"],
            "name": resume_row["name"],
            "pronouns": resume_row["pronouns"],
            "university": resume_row["university"],
            "job_id": job_row["job_id"],
            "job_domain": job_row["domain"],
            "job_title": job_row["title"],
            "company_name": job_row["company_name"],
            "sbert_score": score
        })

# Convert results to DataFrame
sbert_scores = pd.DataFrame(results)
print("SBERT scores shape:", sbert_scores.shape)
display(sbert_scores.head())

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

SBERT scores shape: (200, 12)


,resume_id,resume_domain,version,changed_signal,name,pronouns,university,job_id,job_domain,job_title,company_name,sbert_score
0,R1,Software Engineering,original,none,Emily Johnson,she/her,Stanford University,J1,Software Engineering,Software Engineer,GOYT,0.510895
1,R1,Software Engineering,original,none,Emily Johnson,she/her,Stanford University,J2,Finance,Financial Analyst,Aya Healthcare,0.226471
2,R1,Software Engineering,original,none,Emily Johnson,she/her,Stanford University,J3,Marketing,Marketing Coordinator,Corcoran Sawyer Smith,0.342869
3,R1,Software Engineering,original,none,Emily Johnson,she/her,Stanford University,J4,Healthcare,Clinical Data Analyst,Insight Global,0.329924
4,R1,Software Engineering,original,none,Emily Johnson,she/her,Stanford University,J5,Education,Academic Advisor,Kennesaw State University,0.313125


In [5]:
output_path = "ML_FINAL/results/sbert_scores.csv"
sbert_scores.to_csv(output_path, index=False)
print("Saved to:", output_path)

Saved to: ML_FINAL/results/sbert_scores.csv
